# Setup

In [2]:
from pathlib import Path
import os
import glob
import time
import base64

import torch
import gymnasium as gym
import ale_py

from IPython.display import HTML, display

from stable_baselines3 import A2C, DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, VecVideoRecorder

# Register Atari/ALE environments like ALE/Pong-v5
gym.register_envs(ale_py)

# Figure out project root whether notebook is run from /code or project root
cwd = Path.cwd()

if (cwd / "models").exists() and (cwd / "videos").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "models").exists() and (cwd.parent / "videos").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

MODEL_DIR = PROJECT_ROOT / "models"
VIDEO_DIR = PROJECT_ROOT / "videos"

MODEL_DIR.mkdir(exist_ok=True)
VIDEO_DIR.mkdir(exist_ok=True)

ENV_ID = "ALE/Pong-v5"
SEED = 0

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Current working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Model directory:", MODEL_DIR)
print("Video directory:", VIDEO_DIR)
print("Environment:", ENV_ID)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Current working directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\code
Project root: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3
Model directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\models
Video directory: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\videos
Environment: ALE/Pong-v5
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


# Check video packages

In [3]:
try:
    import moviepy
    import imageio
    import imageio_ffmpeg
    
    print("Video packages are installed.")
    print("moviepy:", moviepy.__version__)
    print("imageio:", imageio.__version__)
except Exception as e:
    print("Video package problem:")
    print(e)

Video packages are installed.
moviepy: 2.1.2
imageio: 2.37.3


# Pick algorithm and checkpoint

In [4]:
# Choose algorithm: "A2C" now, "DQN" later
ALGORITHM = "A2C"
#ALGORITHM = "DQN"

# Choose checkpoint steps that exist in your models folder
#CHECKPOINT_STEPS = 50_000
#CHECKPOINT_STEPS = 500_000
#CHECKPOINT_STEPS = 1_000_000
#CHECKPOINT_STEPS = 1_500_000
#CHECKPOINT_STEPS = 2_000_000
#CHECKPOINT_STEPS = 2_000_000
#CHECKPOINT_STEPS = 3_000_000
CHECKPOINT_STEPS = 4_000_000
#CHECKPOINT_STEPS = 5_000_000
#CHECKPOINT_STEPS = 6_000_000

In [5]:
# Video length in environment steps
VIDEO_LENGTH = 8000

# Safety cap so it does not run forever
MAX_SECONDS = 60

# Map algorithm names to SB3 classes
MODEL_CLASSES = {
    "A2C": A2C,
    "DQN": DQN,
}

if ALGORITHM not in MODEL_CLASSES:
    raise ValueError(f"Unknown algorithm: {ALGORITHM}")

ModelClass = MODEL_CLASSES[ALGORITHM]

model_filename = f"{ALGORITHM.lower()}_pong_{CHECKPOINT_STEPS}.zip"
model_path = MODEL_DIR / model_filename

print("Selected algorithm:", ALGORITHM)
print("Selected checkpoint:", CHECKPOINT_STEPS)
print("Model path:", model_path)

if not model_path.exists():
    raise FileNotFoundError(f"Model file not found: {model_path}")

Selected algorithm: A2C
Selected checkpoint: 4000000
Model path: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\models\a2c_pong_4000000.zip


# Load model

In [6]:
model = ModelClass.load(
    model_path,
    device=device
)

print("Loaded model:", model_path.name)
print("Model type:", ALGORITHM)

Loaded model: a2c_pong_4000000.zip
Model type: A2C


In [7]:
# Create a unique prefix for the video filename
video_prefix = f"{ALGORITHM.lower()}_pong_{CHECKPOINT_STEPS}"

# Optional cleanup: delete only old videos from this same checkpoint
for f in glob.glob(str(VIDEO_DIR / f"{video_prefix}*.mp4")):
    os.remove(f)

# Recreate the same preprocessing pipeline used during training
eval_env = make_atari_env(
    ENV_ID,
    n_envs=1,
    seed=SEED,
    wrapper_kwargs=dict(terminal_on_life_loss=False),
)

eval_env = VecFrameStack(eval_env, n_stack=4)

# Wrap environment with video recorder
eval_env = VecVideoRecorder(
    eval_env,
    video_folder=str(VIDEO_DIR),
    record_video_trigger=lambda step: step == 0,
    video_length=VIDEO_LENGTH,
    name_prefix=video_prefix
)

obs = eval_env.reset()

start_time = time.time()
step_count = 0
done = [False]

while (
    step_count < VIDEO_LENGTH
    and (time.time() - start_time) < MAX_SECONDS
    and not done[0]
):
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, done, infos = eval_env.step(action)
    step_count += 1

eval_env.close()

elapsed = time.time() - start_time

print(f"Stopped after {step_count} steps and {elapsed:.1f} seconds.")

video_files = sorted(VIDEO_DIR.glob(f"{video_prefix}*.mp4"))

if video_files:
    video_path = video_files[-1]
    print("Created video:", video_path)
else:
    raise FileNotFoundError("No video file was created.")

MoviePy - Building video c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\videos\a2c_pong_4000000-step-0-to-step-8000.mp4.
MoviePy - Writing video c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\videos\a2c_pong_4000000-step-0-to-step-8000.mp4



MoviePy - Done !
MoviePy - video ready c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\videos\a2c_pong_4000000-step-0-to-step-8000.mp4
Stopped after 619 steps and 2.1 seconds.
Created video: c:\Users\dosan\Desktop\GradLevel\2-Spring 2026\CS264\Projects\StableBaselines3\videos\a2c_pong_4000000-step-0-to-step-8000.mp4


In [8]:
with open(video_path, "rb") as f:
    mp4 = f.read()

data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()

display(HTML(f"""
<div style="width: 100%; overflow-x: auto;">
    <video controls playsinline style="width: 100%; max-width: 480px; height: auto;">
        <source src="{data_url}" type="video/mp4">
    </video>
</div>
"""))